# 📊 SQL Data Analysis – HR Database

This project demonstrates how to build and analyze a relational database using SQL within Google Colab.

The workflow includes:
- Creating database tables using SQL
- Loading CSV data into tables
- Running analytical SQL queries
- Extracting insights using aggregation and filtering

Tools used: Python, SQLite, Pandas, SQL

## Step 1: Upload Dataset Files

In this step, we upload all required CSV files and the SQL script into the Colab environment.

These files contain structured HR data including employees, departments, jobs, and locations.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Employees.csv to Employees.csv
Saving Locations.csv to Locations.csv
Saving JobsHistory.csv to JobsHistory.csv
Saving Jobs.csv to Jobs.csv
Saving Departments.csv to Departments.csv


## Step 2: Create Database and Tables

We create a local SQLite database and define the schema using SQL.

The database includes the following tables:
- EMPLOYEES
- JOB_HISTORY
- JOBS
- DEPARTMENTS
- LOCATIONS

In [2]:
import sqlite3

conn = sqlite3.connect("hr_lab.db")
cur = conn.cursor()

sql_script = """
CREATE TABLE EMPLOYEES (
    EMP_ID CHAR(9) NOT NULL,
    F_NAME VARCHAR(15) NOT NULL,
    L_NAME VARCHAR(15) NOT NULL,
    SSN CHAR(9),
    B_DATE DATE,
    SEX CHAR,
    ADDRESS VARCHAR(30),
    JOB_ID CHAR(9),
    SALARY DECIMAL(10,2),
    MANAGER_ID CHAR(9),
    DEP_ID CHAR(9) NOT NULL,
    PRIMARY KEY (EMP_ID)
);

CREATE TABLE JOB_HISTORY (
    EMPL_ID CHAR(9) NOT NULL,
    START_DATE DATE,
    JOBS_ID CHAR(9) NOT NULL,
    DEPT_ID CHAR(9),
    PRIMARY KEY (EMPL_ID,JOBS_ID)
);

CREATE TABLE JOBS (
    JOB_IDENT CHAR(9) NOT NULL,
    JOB_TITLE VARCHAR(30),
    MIN_SALARY DECIMAL(10,2),
    MAX_SALARY DECIMAL(10,2),
    PRIMARY KEY (JOB_IDENT)
);

CREATE TABLE DEPARTMENTS (
    DEPT_ID_DEP CHAR(9) NOT NULL,
    DEP_NAME VARCHAR(15),
    MANAGER_ID CHAR(9),
    LOC_ID CHAR(9),
    PRIMARY KEY (DEPT_ID_DEP)
);

CREATE TABLE LOCATIONS (
    LOCT_ID CHAR(9) NOT NULL,
    DEP_ID_LOC CHAR(9) NOT NULL,
    PRIMARY KEY (LOCT_ID,DEP_ID_LOC)
);
"""

cur.executescript(sql_script)
conn.commit()

print("HR tables created successfully.")

HR tables created successfully.


## Step 3: Load Data into Tables

In this step, we load the CSV data into the corresponding database tables using Pandas.

Each CSV file is mapped to its respective table and inserted into the database.

In [3]:
import pandas as pd

files_map = {
    "EMPLOYEES": "Employees.csv",
    "JOB_HISTORY": "JobsHistory.csv",
    "JOBS": "Jobs.csv",
    "DEPARTMENTS": "Departments.csv",
    "LOCATIONS": "Locations.csv",
}

for table, file in files_map.items():
    df = pd.read_csv(file, header=None)
    df.columns = [col[1] for col in pd.read_sql_query(f"PRAGMA table_info({table})", conn).values]
    df.to_sql(table, conn, if_exists="append", index=False)
    print(f"Loaded {len(df)} rows into {table}")

Loaded 10 rows into EMPLOYEES
Loaded 10 rows into JOB_HISTORY
Loaded 10 rows into JOBS
Loaded 3 rows into DEPARTMENTS
Loaded 3 rows into LOCATIONS


## Step 4: Verify Data Load

We verify that the data has been successfully loaded into each table by counting the number of rows.

This ensures that all datasets were imported correctly.

In [4]:
for table in files_map.keys():
    print(f"=== {table} ===")
    print(pd.read_sql_query(f"SELECT COUNT(*) AS row_count FROM {table}", conn))
    print()

=== EMPLOYEES ===
   row_count
0         10

=== JOB_HISTORY ===
   row_count
0         10

=== JOBS ===
   row_count
0         10

=== DEPARTMENTS ===
   row_count
0          3

=== LOCATIONS ===
   row_count
0          3



## Step 5: SQL Data Analysis

We now perform SQL queries to analyze the data.

This includes:
- Filtering records
- Sorting results
- Grouping data
- Aggregating values

## Step 5A: Filtering Data

In this section, we filter records using WHERE and LIKE conditions to find specific patterns in the data.

In [5]:
pd.read_sql_query("""
SELECT F_NAME, L_NAME
FROM EMPLOYEES
WHERE ADDRESS LIKE '%Elgin,IL%';
""", conn)

,F_NAME,L_NAME
0,Alice,James
1,Nancy,Allen
2,Ann,Jacob


## Step 5B: Sorting Data

In this section, we sort results using ORDER BY to organize data.

In [6]:
pd.read_sql_query("""
SELECT F_NAME, L_NAME
FROM EMPLOYEES
WHERE B_DATE LIKE '197%';
""", conn)

,F_NAME,L_NAME
0,John,Thomas
1,Alice,James
2,Nancy,Allen
3,Mary,Thomas


In [7]:
pd.read_sql_query("""
SELECT *
FROM EMPLOYEES
WHERE (SALARY BETWEEN 60000 AND 70000) AND DEP_ID = 5;
""", conn)

,EMP_ID,F_NAME,L_NAME,SSN,B_DATE,SEX,ADDRESS,JOB_ID,SALARY,MANAGER_ID,DEP_ID
0,E1004,Santosh,Kumar,123459,1985-07-20,M,"511 Aurora Av, Aurora,IL",400,60000,30002,5
1,E1010,Ann,Jacob,123415,1982-03-30,F,"111 Britany Springs,Elgin,IL",220,70000,30002,5


In [8]:
pd.read_sql_query("""
SELECT F_NAME, L_NAME, DEP_ID
FROM EMPLOYEES
ORDER BY DEP_ID;
""", conn)

,F_NAME,L_NAME,DEP_ID
0,John,Thomas,2
1,Ahmed,Hussain,2
2,Nancy,Allen,2
3,Alice,James,5
4,Steve,Wells,5
5,Santosh,Kumar,5
6,Ann,Jacob,5
7,Mary,Thomas,7
8,Bharath,Gupta,7
9,Andrea,Jones,7


In [9]:
pd.read_sql_query("""
SELECT F_NAME, L_NAME, DEP_ID
FROM EMPLOYEES
ORDER BY DEP_ID DESC, L_NAME DESC;
""", conn)

,F_NAME,L_NAME,DEP_ID
0,Mary,Thomas,7
1,Andrea,Jones,7
2,Bharath,Gupta,7
3,Steve,Wells,5
4,Santosh,Kumar,5
5,Alice,James,5
6,Ann,Jacob,5
7,John,Thomas,2
8,Ahmed,Hussain,2
9,Nancy,Allen,2


## Step 5C: Grouping Data

We group records by department to summarize employee distribution.

In [10]:
pd.read_sql_query("""
SELECT DEP_ID, COUNT(*)
FROM EMPLOYEES
GROUP BY DEP_ID;
""", conn)

,DEP_ID,COUNT(*)
0,2,3
1,5,4
2,7,3


## Step 5D: Aggregating Data

We use aggregate functions such as COUNT() and AVG() to calculate insights like total employees and average salary.

In [11]:
pd.read_sql_query("""
SELECT DEP_ID, COUNT(*) AS NUM_EMPLOYEES, AVG(SALARY) AS AVG_SALARY
FROM EMPLOYEES
GROUP BY DEP_ID;
""", conn)

,DEP_ID,NUM_EMPLOYEES,AVG_SALARY
0,2,3,86666.666667
1,5,4,65000.000000
2,7,3,66666.666667


In [12]:
pd.read_sql_query("""
SELECT DEP_ID, COUNT(*) AS NUM_EMPLOYEES, AVG(SALARY) AS AVG_SALARY
FROM EMPLOYEES
GROUP BY DEP_ID
ORDER BY AVG_SALARY;
""", conn)

,DEP_ID,NUM_EMPLOYEES,AVG_SALARY
0,5,4,65000.000000
1,7,3,66666.666667
2,2,3,86666.666667


## Step 5E: Filtering Grouped Data

The HAVING clause filters grouped results based on aggregate conditions.

In [13]:
pd.read_sql_query("""
SELECT DEP_ID, COUNT(*) AS NUM_EMPLOYEES, AVG(SALARY) AS AVG_SALARY
FROM EMPLOYEES
GROUP BY DEP_ID
HAVING COUNT(*) < 4
ORDER BY AVG_SALARY;
""", conn)

,DEP_ID,NUM_EMPLOYEES,AVG_SALARY
0,7,3,66666.666667
1,2,3,86666.666667


In [14]:
pd.read_sql_query("""
SELECT *
FROM EMPLOYEES
WHERE F_NAME LIKE 'S%';
""", conn)

,EMP_ID,F_NAME,L_NAME,SSN,B_DATE,SEX,ADDRESS,JOB_ID,SALARY,MANAGER_ID,DEP_ID
0,E1003,Steve,Wells,123458,1980-10-08,M,"291 Springs, Gary,IL",300,50000,30002,5
1,E1004,Santosh,Kumar,123459,1985-07-20,M,"511 Aurora Av, Aurora,IL",400,60000,30002,5


In [15]:
pd.read_sql_query("""
SELECT *
FROM EMPLOYEES
ORDER BY B_DATE;
""", conn)

,EMP_ID,F_NAME,L_NAME,SSN,B_DATE,SEX,ADDRESS,JOB_ID,SALARY,MANAGER_ID,DEP_ID
0,E1002,Alice,James,123457,1972-07-31,F,"980 Berry ln, Elgin,IL",200,80000,30002,5
1,E1007,Mary,Thomas,123412,1975-05-05,F,"100 Rose Pl, Gary,IL",650,65000,30003,7
2,E1001,John,Thomas,123456,1976-09-01,M,"5631 Rice, OakPark,IL",100,100000,30001,2
3,E1006,Nancy,Allen,123411,1978-06-02,F,"111 Green Pl, Elgin,IL",600,90000,30001,2
4,E1003,Steve,Wells,123458,1980-10-08,M,"291 Springs, Gary,IL",300,50000,30002,5
5,E1005,Ahmed,Hussain,123410,1981-04-01,M,"216 Oak Tree, Geneva,IL",500,70000,30001,2
6,E1010,Ann,Jacob,123415,1982-03-30,F,"111 Britany Springs,Elgin,IL",220,70000,30002,5
7,E1008,Bharath,Gupta,123413,1985-06-05,M,"145 Berry Ln, Naperville,IL",660,65000,30003,7
8,E1004,Santosh,Kumar,123459,1985-07-20,M,"511 Aurora Av, Aurora,IL",400,60000,30002,5
9,E1009,Andrea,Jones,123414,1990-09-07,F,"120 Fall Creek, Gary,IL",234,70000,30003,7


In [16]:
pd.read_sql_query("""
SELECT DEP_ID, AVG(SALARY) AS AVG_SALARY
FROM EMPLOYEES
GROUP BY DEP_ID
HAVING AVG(SALARY) >= 60000;
""", conn)

,DEP_ID,AVG_SALARY
0,2,86666.666667
1,5,65000.000000
2,7,66666.666667


In [17]:
pd.read_sql_query("""
SELECT DEP_ID, AVG(SALARY) AS AVG_SALARY
FROM EMPLOYEES
GROUP BY DEP_ID
HAVING AVG(SALARY) >= 60000
ORDER BY AVG_SALARY DESC;
""", conn)

,DEP_ID,AVG_SALARY
0,2,86666.666667
1,7,66666.666667
2,5,65000.000000


## Conclusion

In this lab, I successfully:

- Created and structured relational database tables using SQL
- Loaded and verified data from CSV files
- Performed data analysis using SQL queries within Python

Key SQL concepts applied:

- Filtering data using WHERE and LIKE
- Sorting results using ORDER BY
- Grouping data using GROUP BY
- Aggregating data using COUNT() and AVG()
- Filtering grouped results using HAVING

This exercise demonstrates how SQL can be integrated with Python to perform efficient data analysis on structured datasets.

In [18]:
conn.close()